# STAC data prep — GEBCO 2025 grid (4 COG tiles)

Notebook to migrate the four GEBCO 2025 (sub-ice topography/bathymetry) `.tif` tiles to **Cloud Optimized GeoTIFFs (CoG)**.
Same pattern as `17_seabed_litho.ipynb` and `20_groundwater_recharge.ipynb`, with one COG per source tile (a grid of COGs, see `DATA_FORMATS.md`, "COG tiles") and one STAC item per tile.

- **Case:** 2D rasters, continuous elevation in metres (negative = bathymetry), 15 arc-second grid (0.004167 degrees), `int16`. Each tile is 21600 x 21600 pixels (about 0.93 GB uncompressed) and covers 90 x 90 degrees; together they cover lon -90..90, lat -90..90.
- **Format:** one COG per tile, no clipping. The tiles are not merged: a single 43200 x 43200 raster would be unwieldy to produce, upload and publish.
- **Source data:** the `gebco_2025_sub_ice_*` tiles (lossless `int16`). `GEBCO_Africa_merged.tif` is not used: it is the same window converted to `float32` (7.5 GB), and it is not clipped to Africa.
- **Resampling:** `AVERAGE` for overviews, because elevation is continuous.
- **CRS:** the source tiles carry no CRS tag; they are written as EPSG:4326 (GEBCO is a WGS84 geographic grid).
- **Metadata template:** `metadata/metadata_GEBCO.json`. The template mentions GEBCO_2026, but the data are the **GEBCO_2025** grid: version, description, citation and DOI are corrected below.

Sources (`P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian`): `gebco_2025_sub_ice_n0.0_s-90.0_w-90.0_e0.0.tif` and the three other tiles.

**Running locally (recommended for this dataset):** with `DATA_VIVIAN`/`STAC_DATA_REPO` unset, the notebook reads and writes `P:` directly. Run it in the `gca-exploration` env (kernel `gca-exploration`); no copy to a local mirror is needed. The docker route below also works but copies about 3.7 GB each way:

```powershell
docker\run_notebook_with_p_drive.ps1 -Notebook 21_gebco.ipynb -Variable gebco -InputPattern "gebco_2025*.tif"
```

### Import packages

In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import rasterio
import rioxarray as rxr

### Define drive paths

`DATA_VIVIAN` and `STAC_DATA_REPO` are set in the docker container; otherwise the Windows host paths are used.
The item id is the file name without `.tif` and without the decimals (`..._n0_s-90_w-90_e0`): dots in an id would be treated as a file suffix by the STAC layout.

In [2]:
COLLECTION_ID = "gebco"
OUT_NODATA = -32767  # nodata declared by the GEBCO tiles

# Same layout as P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian (DATA_VIVIAN is set in docker,
# where the P: drive is not mounted; a local mirror of that folder is used and the stac/ output is copied back to P:)
data_vivian = Path(os.environ.get("DATA_VIVIAN", r"P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

def item_id_from(path: Path) -> str:
    return re.sub(r"(\d)\.0(?=_|$)", r"\1", path.stem)  # n0.0_s-90.0 -> n0_s-90

source_tifs = {item_id_from(p): p for p in sorted(data_vivian.glob("gebco_2025_sub_ice_*.tif"))}
if len(source_tifs) != 4:
    raise FileNotFoundError(f"Expected 4 gebco_2025_sub_ice_*.tif tiles in {data_vivian}, found {len(source_tifs)}")
metadata_template_path = stac_repo / "metadata" / "metadata_GEBCO.json"
if not metadata_template_path.is_file():
    raise FileNotFoundError(metadata_template_path)

processed_data_dir = data_vivian / "stac" / COLLECTION_ID
cog_dir = processed_data_dir / "cog"
cog_dir.mkdir(parents=True, exist_ok=True)

for item_id, p in source_tifs.items():
    print(f"{item_id}  <-  {p.name}")
print("Metadata template:", metadata_template_path)
print("Output dir:", processed_data_dir)

gebco_2025_sub_ice_n0_s-90_w-90_e0  <-  gebco_2025_sub_ice_n0.0_s-90.0_w-90.0_e0.0.tif
gebco_2025_sub_ice_n0_s-90_w0_e90  <-  gebco_2025_sub_ice_n0.0_s-90.0_w0.0_e90.0.tif
gebco_2025_sub_ice_n90_s0_w-90_e0  <-  gebco_2025_sub_ice_n90.0_s0.0_w-90.0_e0.0.tif
gebco_2025_sub_ice_n90_s0_w0_e90  <-  gebco_2025_sub_ice_n90.0_s0.0_w0.0_e90.0.tif
Metadata template: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\metadata\metadata_GEBCO.json
Output dir: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\gebco


### Read raw data

Only the headers are read here; each tile is loaded one at a time when it is written.

In [3]:
tile_info = {}
for item_id, path in source_tifs.items():
    with rasterio.open(path) as src:
        b = src.bounds
        tile_info[item_id] = {
            "bbox": [round(v, 6) for v in (b.left, b.bottom, b.right, b.top)],
            "shape": src.shape,
            "res": src.res,
        }
        print(f"{item_id}: {src.width} x {src.height} | dtype {src.dtypes[0]} | nodata {src.nodata} | crs {src.crs} | "
              f"bbox {tile_info[item_id]['bbox']} | res {src.res[0]:.6f} | tiled {src.is_tiled} | overviews {src.overviews(1)}")
        assert src.dtypes[0] == "int16" and src.nodata == OUT_NODATA

# The four tiles must form the lon -90..90 / lat -90..90 window without gaps or overlaps
bboxes = np.array([t["bbox"] for t in tile_info.values()])
assert np.allclose(sorted({*bboxes[:, 0], *bboxes[:, 2]}), [-90, 0, 90], atol=1e-6)
assert np.allclose(sorted({*bboxes[:, 1], *bboxes[:, 3]}), [-90, 0, 90], atol=1e-6)
print("OK: 4 tiles, 2 x 2 grid")

gebco_2025_sub_ice_n0_s-90_w-90_e0: 21600 x 21600 | dtype int16 | nodata -32767.0 | crs EPSG:4326 | bbox [-90.0, -90.0, 0.0, 0.0] | res 0.004167 | tiled False | overviews []
gebco_2025_sub_ice_n0_s-90_w0_e90: 21600 x 21600 | dtype int16 | nodata -32767.0 | crs EPSG:4326 | bbox [-0.0, -90.0, 90.0, 0.0] | res 0.004167 | tiled False | overviews []
gebco_2025_sub_ice_n90_s0_w-90_e0: 21600 x 21600 | dtype int16 | nodata -32767.0 | crs EPSG:4326 | bbox [-90.0, 0.0, 0.0, 90.0] | res 0.004167 | tiled False | overviews []
gebco_2025_sub_ice_n90_s0_w0_e90: 21600 x 21600 | dtype int16 | nodata -32767.0 | crs EPSG:4326 | bbox [-0.0, 0.0, 90.0, 90.0] | res 0.004167 | tiled False | overviews []
OK: 4 tiles, 2 x 2 grid


### Check CF compliancy original file

Not applicable for a plain GeoTIFF (same approach as `11b_Salinity` and `17_seabed_litho`): CRS, nodata and attributes are written explicitly below.

### Make CF compliant alterations (dataset dependent)

Fill the metadata fields that are empty (or wrong) in the template: version GEBCO_2025 (not 2026), description, citation, `SPATIAL_EXTENT`, `TEMPORAL_EXTENT`, `SPATIAL_RESOLUTION`, `NODATA`, `DATA_TYPE`, `DIMENSIONS`, `MEDIA_TYPE`, `DATA_MODEL`, `COLLECTION_ID`, `UNITS`, ...
`ITEMS` holds the per-tile title, description and bbox for the STAC builder.

In [4]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

def tile_title(bbox):
    w, s, e, n = bbox
    return f"GEBCO 2025 grid, tile lon {w:g}..{e:g}, lat {s:g}..{n:g}"

items_meta = {
    item_id: {
        "TITLE": tile_title(info["bbox"]),
        "DESCRIPTION": f"GEBCO_2025 Grid (sub-ice topography/bathymetry), 15 arc-second tile covering lon {info['bbox'][0]:g} to {info['bbox'][2]:g} and lat {info['bbox'][1]:g} to {info['bbox'][3]:g}.",
        "BBOX": info["bbox"],
    }
    for item_id, info in tile_info.items()
}

metadata.update({
    "TITLE": "General Bathymetric Chart of the Oceans (GEBCO 2025 Grid)",
    "TITLE_ABBREVIATION": "GEBCO",
    "DESCRIPTION": (
        "GEBCO's gridded bathymetric data set, the GEBCO_2025 Grid, is a global terrain model for ocean and land, "
        "providing elevation data, in metres, on a 15 arc-second interval grid. This is the sub-ice topography/bathymetry "
        "version (under-ice topography for Greenland and Antarctica), delivered as four 90 x 90 degree Cloud Optimized GeoTIFF tiles "
        "covering longitude -90 to 90 and latitude -90 to 90."
    ),
    "SHORT_DESCRIPTION": "Global terrain data (elevation and bathymetry, m) for land and ocean, GEBCO_2025 grid",
    "HISTORY": [
        "GEBCO_2025 Grid (sub-ice topo/bathy), published August 2025, downloaded as four 90 x 90 degree GeoTIFF tiles",
        "Each tile converted to a Cloud Optimized GeoTIFF (DEFLATE, predictor, average overviews); values unchanged",
    ],
    "MEDIA_TYPE": "image/tiff; application=geotiff; profile=cloud-optimized",
    "DATA_MODEL": "raster_cog_tiles",
    "DIMENSIONS": ["x", "y"],
    "SPATIAL_EXTENT": [-90.0, -90.0, 90.0, 90.0],
    "TEMPORAL_EXTENT": ["2025-01-01T00:00:00Z", "2025-12-31T23:59:59Z"],
    "KEYWORDS": ["bathymetry", "topography", "elevation", "terrain", "ocean", "seabed", "GEBCO", "Seabed 2030"],
    "TAGS": ["bathymetry", "elevation", "gebco", "cog"],
    "CITATION": "GEBCO Compilation Group (2025) GEBCO 2025 Grid (doi:10.5285/37c52e96-24ea-67ce-e063-7086abc05f29)",
    "DOI": "https://doi.org/10.5285/37c52e96-24ea-67ce-e063-7086abc05f29",
    "LONG_NAME": "Elevation relative to sea level",
    "UNITS": "m",
    "COMMENT": (
        "GEBCO_2025 Grid (sub-ice topo/bathy). The four source tiles (e.g. gebco_2025_sub_ice_n0.0_s-90.0_w0.0_e90.0) are kept "
        "as four COGs, one STAC item each; negative values are ocean depth, positive values are land elevation. "
        f"{OUT_NODATA} is the nodata value declared by the source files. The values are pixel-centre registered. "
        "The metadata template referred to GEBCO_2026; the data are GEBCO_2025 and the version, citation and DOI were corrected. "
        "The merged file GEBCO_Africa_merged.tif is not used (float32 copy of the same window, not clipped to Africa)."
    ),
    "CRS": "EPSG:4326",
    "ITEM_BBOX_CRS": "EPSG:4326",
    "SPATIAL_RESOLUTION": round(1 / 240, 6),  # 15 arc-seconds
    "NODATA": OUT_NODATA,
    "DATA_TYPE": "int16",
    "COLLECTION_ID": COLLECTION_ID,
    "WMS_DATASET": COLLECTION_ID,  # GeoServer workspace used by scripts/21_gebco.ipynb
    "ITEMS": items_meta,
})

print("TITLE:", metadata["TITLE"])
print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"], "| SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("License in the template:", metadata["LICENSE"])

TITLE: General Bathymetric Chart of the Oceans (GEBCO 2025 Grid)
SPATIAL_EXTENT: [-90.0, -90.0, 90.0, 90.0] | SPATIAL_RESOLUTION: 0.004167
License in the template: Creative Commons Attribution 4.0


### Write data to CoG

One CoG per tile, `DEFLATE` compression with the integer predictor, **average** overviews because elevation is continuous. Tiles are processed one at a time to keep memory below about 3 GB.

In [5]:
# Dataset attributes are written as GeoTIFF tags (scalars only)
attrs = {}
for attr_name, attr_val in metadata.items():
    if attr_name == "ITEMS":
        continue
    if attr_name == "MEDIA_TYPE":
        attr_val = "IMAGE/TIFF"  # same convention as 11b_Salinity
    if isinstance(attr_val, (list, dict)):
        attr_val = json.dumps(attr_val)
    attrs[attr_name] = attr_val
attrs["Conventions"] = "CF-1.8"

out_paths = {}
for item_id, path in source_tifs.items():
    ds = rxr.open_rasterio(path, mask_and_scale=False)
    da = ds.isel(band=0).drop_vars("band").rio.write_crs(4326).rio.write_nodata(OUT_NODATA)
    da.attrs.update(attrs)
    da.attrs.update({"_FillValue": OUT_NODATA, "TITLE": items_meta[item_id]["TITLE"]})
    da.name = item_id

    out_paths[item_id] = cog_dir / f"{item_id}.tif"
    da.rio.to_raster(
        out_paths[item_id],
        driver="COG",
        compress="DEFLATE",
        predictor="YES",
        dtype="int16",
        overview_resampling="AVERAGE",
        resampling="NEAREST",
        num_threads="ALL_CPUS",
    )
    print(f"Wrote COG: {out_paths[item_id].name} ({out_paths[item_id].stat().st_size / 1e6:.0f} MB, source {path.stat().st_size / 1e6:.0f} MB)")
    del ds, da

metadata_output_path = processed_data_dir / f"metadata_{COLLECTION_ID}.json"
with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)
print("Wrote metadata:", metadata_output_path)

Wrote COG: gebco_2025_sub_ice_n0_s-90_w-90_e0.tif (396 MB, source 933 MB)
Wrote COG: gebco_2025_sub_ice_n0_s-90_w0_e90.tif (380 MB, source 933 MB)
Wrote COG: gebco_2025_sub_ice_n90_s0_w-90_e0.tif (444 MB, source 933 MB)
Wrote COG: gebco_2025_sub_ice_n90_s0_w0_e90.tif (406 MB, source 933 MB)
Wrote metadata: P:\11209117-041-ipdc-egypt-swi\data\IntDeltaPlatform_QGA\Data_Vivian\stac\gebco\metadata_gebco.json


### Verify the output

Every pixel of every COG must equal the source tile (the conversion is lossless).

In [6]:
for item_id, out_path in out_paths.items():
    with rasterio.open(out_path) as src:
        assert src.dtypes[0] == "int16"
        assert src.nodata == OUT_NODATA
        assert src.crs.to_epsg() == 4326
        assert src.is_tiled, "COG must be internally tiled"
        assert len(src.overviews(1)) > 0, "COG must have overviews"
        assert [round(v, 6) for v in src.bounds] == tile_info[item_id]["bbox"], "Bounds changed"
        new = src.read(1)
        print(f"{item_id}: block {src.profile.get('blockxsize')} x {src.profile.get('blockysize')} | "
              f"{len(src.overviews(1))} overviews | compression {src.compression}")

    with rasterio.open(source_tifs[item_id]) as src:
        old = src.read(1)
    assert np.array_equal(old, new), "Pixel values changed during COG conversion"
    print(f"   identical to source | min {new.min()} m | max {new.max()} m | nodata pixels {(new == OUT_NODATA).sum()}")
    del old, new
print("OK: GEBCO COG tiles and metadata are ready for the STAC builder notebook.")

gebco_2025_sub_ice_n0_s-90_w-90_e0: block 512 x 512 | 6 overviews | compression Compression.deflate
   identical to source | min -8976 m | max 6787 m | nodata pixels 0
gebco_2025_sub_ice_n0_s-90_w0_e90: block 512 x 512 | 6 overviews | compression Compression.deflate
   identical to source | min -7185 m | max 5810 m | nodata pixels 0
gebco_2025_sub_ice_n90_s0_w-90_e0: block 512 x 512 | 6 overviews | compression Compression.deflate
   identical to source | min -8733 m | max 5654 m | nodata pixels 0
gebco_2025_sub_ice_n90_s0_w0_e90: block 512 x 512 | 6 overviews | compression Compression.deflate
   identical to source | min -8149 m | max 8627 m | nodata pixels 0
OK: GEBCO COG tiles and metadata are ready for the STAC builder notebook.
